# SME Bank Statement Analyzer — cash-flow analysis from bank statements

The LLM model step runs only when `USE_MODEL = True` and the rules left transactions unclassified; every call spends credit.

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

for lib_dir in ("/opt/homebrew/lib", "/usr/local/lib"):
    if Path(lib_dir).is_dir():
        os.environ.setdefault("DYLD_FALLBACK_LIBRARY_PATH", lib_dir)
        break

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=True)

TESTCASE_ID = "case_1"
TAX_CODE = "9999000002"
ANALYSIS_MONTH = "" # "YYYY-MM" for month T; empty = month of the last transaction
USE_MODEL = True # True: call the model for transactions the rules left unclassified
MANUAL_TRADE_ASSESSMENT = "" # officer override for indicator 6, one of the report values: "Phù hợp" / "Có chênh lệch giải thích được" / "Không phù hợp đáng kể"
CASE_DIR = PROJECT_ROOT / "samples" / TESTCASE_ID

In [ ]:
from src.config import Config, build_llm
from src.tools._executor import sqlite_executor

config = Config(
    query_executor=sqlite_executor(str(PROJECT_ROOT / "samples" / "dummy_db" / "bankstatementanalyzer_dummy.sqlite")),
    classifier_llm=build_llm("MODEL_CLASSIFIER") if USE_MODEL else None,
    max_classifier_calls=6,
    patterns_per_call=80,
    manual_trade_assessment=MANUAL_TRADE_ASSESSMENT,
)

In [ ]:
from src.agents.supervisor import run_analysis

result = run_analysis(CASE_DIR, config, TAX_CODE, ANALYSIS_MONTH)
print("\n".join(result.steps))
print("Coverage:", result.analysis["coverage"]["conclusion"],
      "| Revenue reliability:", result.analysis["score"]["rating"])

In [ ]:
from IPython.display import Markdown, display
from src.utils.report.export import write_outputs

out_dir = PROJECT_ROOT / "logs" / f"{TESTCASE_ID}_{datetime.now():%Y%m%d_%H%M%S}"
for name, path in write_outputs(result, out_dir).items():
    print(f"{name}: {path}")
display(Markdown(result.report_markdown))